# CSE6242 - HW3 - Q1

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove any comment that says "#export" because that will crash the autograder in Gradescope. We use this comment to export your code in these cells for grading.
</div>

Pyspark Imports

In [1]:
#export
### DO NOT MODIFY THIS CELL ###
import pyspark
from pyspark.sql import SQLContext
from pyspark.sql.functions import hour, when, col, date_format, to_timestamp, ceil, coalesce

Initialize PySpark Context

In [2]:
### DO NOT MODIFY THIS CELL ###
sc = pyspark.SparkContext(appName="HW3-Q1")
sqlContext = SQLContext(sc)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 04:10:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
/usr/local/lib/python3.9/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Define function for loading data

In [3]:
### DO NOT MODIFY THIS CELL ###
def load_data():
    df = sqlContext.read.option("header",True) \
     .csv("yellow_tripdata_2019-01_short.csv")
    return df

### Q1.1

Perform data casting to clean incoming dataset

In [16]:
#export
def clean_data(df):
    '''
    input: df a dataframe
    output: df a dataframe with the all the original columns
    '''
    
    # START YOUR CODE HERE ---------

    from pyspark.sql.types import IntegerType, FloatType, TimestampType
    
    df = df.withColumn('passenger_count', col('passenger_count').cast(IntegerType())) \
            .withColumn('total_amount', col('total_amount').cast(FloatType())) \
            .withColumn('tip_amount', col('tip_amount').cast(FloatType())) \
            .withColumn('trip_distance', col('trip_distance').cast(FloatType())) \
            .withColumn('fare_amount', col('fare_amount').cast(FloatType())) \
            .withColumn('tpep_pickup_datetime', col('tpep_pickup_datetime').cast(TimestampType())) \
            .withColumn('tpep_dropoff_datetime', col('tpep_dropoff_datetime').cast(TimestampType()))

    # print(df.head(1))
    
    # END YOUR CODE HERE -----------
    
    return df

### Q1.2

Find rate per person for based on how many passengers travel between pickup and dropoff locations. 

In [5]:
#export
def common_pair(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - PULocationID
            - DOLocationID
            - total_passenger_count
            - per_person_rate
            
    per_person_rate is the total_amount per person for a given pair.
    
    '''
    
    # START YOUR CODE HERE ---------
    
    from pyspark.sql.functions import sum
    
    df = df.filter(col('PULocationID') != col('DOLocationID'))

    df= df.groupBy('PULocationID', 'DOLocationID').agg(
        sum('passenger_count').alias('total_passenger_count'),
        (sum('total_amount') / sum('passenger_count')).alias('per_person_rate')
    )

    df = df.orderBy(
        col('total_passenger_count').desc(),
        col('per_person_rate').desc()
    )

    df = df.limit(10)
    
    # END YOUR CODE HERE -----------
    
    return df

### Q1.3

Find trips which trip distances generate the highest tip percentage.

In [15]:
#export
def distance_with_most_tip(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - trip_distance
            - tip_percent
            
    trip_percent is the percent of tip out of fare_amount
    
    '''
    
    # START YOUR CODE HERE ---------

    from pyspark.sql.functions import avg
    
    df = df.filter((col('fare_amount') > 2.00) & (col('trip_distance') > 0))

    df = df.withColumn('tip_percent', (col('tip_amount') * 100) / col('fare_amount')) \
        .withColumn('trip_distance', ceil(col('trip_distance')))
    
    df = df.groupBy('trip_distance').agg(
        avg('tip_percent').alias('tip_percent')
    )

    df = df.orderBy(col('tip_percent').desc()).limit(15)

    
    # END YOUR CODE HERE -----------
    
    return df

### Q1.4

Determine the average speed at different times of day.

In [12]:
#export
def time_with_most_traffic(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - time_of_day
            - am_avg_speed
            - pm_avg_speed
            
    am_avg_speed and pm_avg_speed are the average trip distance / average trip time calculated for each hour
    
    '''
    
    # START YOUR CODE HERE ---------

    from pyspark.sql.functions import avg, first
    
    df = df.withColumn('duration_hours', (col('tpep_dropoff_datetime').cast('long') - col('tpep_pickup_datetime').cast('long')) / 3600.0)
    
    df = df.withColumn("time_of_day", date_format(col("tpep_pickup_datetime"), "K").cast("int"))

    df = df.withColumn("period", 
        when(hour(col("tpep_pickup_datetime")) < 12, "am_avg_speed")
        .otherwise("pm_avg_speed"))
    
    df_agg = df.groupBy("time_of_day", "period").agg(
        (avg("trip_distance") / avg("duration_hours")).alias("speed")
    )

    df = df_agg.groupBy("time_of_day") \
               .pivot("period", ["am_avg_speed", "pm_avg_speed"]) \
               .agg(first("speed"))

    df = df.orderBy("time_of_day")
        
    # END YOUR CODE HERE -----------
    
    return df

## The below cells are for you to investigate your solutions and will not be graded

In [8]:
df = load_data()
df = clean_data(df)

In [9]:
common_pair(df).show()

+------------+------------+---------------------+------------------+
|PULocationID|DOLocationID|total_passenger_count|   per_person_rate|
+------------+------------+---------------------+------------------+
|         239|         238|                   62|  4.26274198870505|
|         237|         236|                   60| 4.482500068346659|
|         263|         141|                   52|3.4190384974846473|
|         161|         236|                   42| 5.368571440378825|
|         148|          79|                   42| 4.711904752822149|
|         142|         238|                   39|  5.05487182812813|
|         141|         236|                   37| 4.355675723101641|
|         239|         143|                   37| 4.252162224537617|
|         239|         142|                   35| 3.817714350564139|
|          79|         170|                   34| 6.394705884596881|
+------------+------------+---------------------+------------------+



In [10]:
distance_with_most_tip(df).show()

+-------------+------------------+
|trip_distance|       tip_percent|
+-------------+------------------+
|            2|15.815527155632552|
|           17|15.796441782308916|
|           20| 15.11240992123345|
|            1| 14.99150513833877|
|            3|14.886705727113446|
|            6|14.579695131601051|
|            5|14.245405861990653|
|            4|13.831569507473274|
|            9|13.814476557648435|
|            8|12.072596772433315|
|           19|11.952632334985276|
|           10|11.880490518902954|
|            7| 10.80057562837643|
|           21|10.739019886973427|
|           18|10.696822158448429|
+-------------+------------------+



In [13]:
time_with_most_traffic(df).show()

+-----------+------------------+------------------+
|time_of_day|      am_avg_speed|      pm_avg_speed|
+-----------+------------------+------------------+
|          0| 9.377696197863992|              NULL|
|          1|10.845483498407583| 5.125214408233276|
|          3|              NULL|               0.0|
|          4|              NULL|               0.0|
|          5|              NULL|0.5137660219476305|
|          6|              NULL|  9.98984771573604|
|          7|              NULL|0.1841530573490315|
|          8|              NULL|0.5183127584049215|
|         10|              NULL|0.6147483852980808|
|         11|              NULL| 4.650958301847908|
+-----------+------------------+------------------+

